In [8]:
pip install pyspark

In [9]:
# ============================================
# MOUNT GOOGLE DRIVE
# ============================================

from google.colab import drive

drive.mount('/content/drive')

# role:
# connects Google Colab with Google Drive

Mounted at /content/drive


In [14]:
# ============================================
# IMPORT SPARK LIBRARIES
# ============================================

from pyspark.sql import SparkSession

from pyspark.sql.functions import *

# role:
# imports Spark processing functions


# ============================================
# CREATE SPARK SESSION
# ============================================

spark = SparkSession.builder \
    .appName("RealTimeLogAnalytics") \
    .getOrCreate()

# role:
# initializes Spark engine


spark.sparkContext.setLogLevel("ERROR")

# role:
# cleaner terminal logs


# ============================================
# READ JSON LOG FILES
# ============================================

df = spark.read.json(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/logs/*.json"

)

# role:
# reads JSON logs
# Spark automatically infers schema


# ============================================
# SHOW INPUT DATA
# ============================================

print("RAW INPUT DATA")

df.show(truncate=False)

# role:
# verifies input logs


print("RAW RECORD COUNT")

print(df.count())

# role:
# verifies total records


# ============================================
# CONVERT TIMESTAMP
# ============================================

df = df.withColumn(

    "timestamp",

    to_timestamp(
        col("time-stamp"),
        "yyyy-MM-dd'T'HH:mm:ss"
        )

)

# role:
# converts timestamp string into Spark timestamp datatype


# ============================================
# BRONZE LAYER
# ============================================

df.write.mode("overwrite").csv(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/bronze",

    header=True

)

# role:
# stores raw logs
# ingestion layer


print("Bronze layer completed")


# ============================================
# SILVER LAYER
# ============================================

silver_df = df.filter(

    col("status_codes").isNotNull()

)

# role:
# removes invalid records


print("SILVER DATA")

silver_df.show(truncate=False)


print("SILVER RECORD COUNT")

print(silver_df.count())


silver_df.write.mode("overwrite").csv(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/silver",

    header=True

)

# role:
# stores cleaned reliable logs


print("Silver layer completed")


# ============================================
# GOLD LAYER
# ============================================

gold_df = silver_df.groupBy(

    "service"

).agg(

    avg("response_time").alias(

        "avg_response_time"

    ),

    # role:
    # calculates average API latency


    count("*").alias(

        "request_count"

    ),

    # role:
    # counts total API requests


    sum(

        when(

            col("status_codes") >= 500,

            1

        ).otherwise(0)

    ).alias(

        "error_count"

    )

    # role:
    # calculates total server failures

)

# role:
# creates business KPIs


print("GOLD DATA")

gold_df.show(truncate=False)


gold_df.write.mode("overwrite").csv(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/gold",

    header=True

)

# role:
# stores analytics-ready aggregated data


print("Gold layer completed")


# ============================================
# VERIFY BRONZE OUTPUT
# ============================================

bronze_df = spark.read.csv(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/bronze",

    header=True

)

print("BRONZE RECORD COUNT")

print(bronze_df.count())


# ============================================
# VERIFY SILVER OUTPUT
# ============================================

silver_check_df = spark.read.csv(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/silver",

    header=True

)

print("SILVER RECORD COUNT")

print(silver_check_df.count())


# ============================================
# VERIFY GOLD OUTPUT
# ============================================

gold_check_df = spark.read.csv(

    "/content/drive/MyDrive/Colab Notebooks/log_analytics/gold",

    header=True

)

print("SILVE OUTPUT")

silver_df.show(truncate=False)

print("GOLD OUTPUT")

gold_check_df.show(truncate=False)


print("PROJECT COMPLETED SUCCESSFULLY")

RAW INPUT DATA
+---------+---------+---------------+------------+-------+-------------+-----------------+------------+-------------------+
|cpu_usage|endpoint |ip_address     |memory_usage|message|response_time|service          |status_codes|time-stamp         |
+---------+---------+---------------+------------+-------+-------------+-----------------+------------+-------------------+
|17       |/checkout|103.24.222.220 |27          |success|2622         |search-service   |200         |2026-05-25T14:33:18|
|27       |/products|138.74.66.218  |78          |success|2440         |payment-service  |200         |2026-05-25T14:33:32|
|77       |/checkout|95.250.122.152 |20          |Error  |400          |shipping-service |404         |2026-05-25T14:33:08|
|95       |/products|112.64.139.9   |23          |success|4809         |payment-service  |200         |2026-05-25T14:33:10|
|12       |/products|155.136.221.95 |44          |Error  |1752         |payment-service  |500         |2026-05-25T14:

In [ ]:
pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00


In [5]:
import os
import boto3
from dotenv import load_dotenv

In [ ]:
s3 = boto3.client(
    's3',
    aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name="ap-south-1"
)

In [ ]:
bronze_path = "/content/drive/MyDrive/Colab Notebooks/log_analytics/bronze"
bucket_name = "real-time-log-analytics"

In [ ]:
for file in os.listdir(bronze_path):
  local_file = os.path.join(
      bronze_path,
      file
  )

  if os.path.isfile(local_file):
    s3.upload_file(
        local_file,
        bucket_name,
        f"bronze/{file}"
    )
    print(f"uploaded{file}")

uploadedpart-00000-474bdb4a-4979-4217-9b8d-3ed09cb1c73e-c000.csv
uploadedpart-00001-474bdb4a-4979-4217-9b8d-3ed09cb1c73e-c000.csv
uploaded.part-00001-474bdb4a-4979-4217-9b8d-3ed09cb1c73e-c000.csv.crc
uploaded.part-00000-474bdb4a-4979-4217-9b8d-3ed09cb1c73e-c000.csv.crc
uploaded_SUCCESS
uploaded._SUCCESS.crc


In [ ]:
silver_path = "/content/drive/MyDrive/Colab Notebooks/log_analytics/silver"

for file in os.listdir(silver_path):
  local_file = os.path.join(
      silver_path,
      file
  )

  if os.path.isfile(local_file):
    s3.upload_file(
        local_file,
        bucket_name,
        f"silver/{file}"
    )
    print(f"uploaded{file}")

uploadedpart-00000-b538a6e6-fa23-4fbe-bf57-6f609930c105-c000.csv
uploadedpart-00001-b538a6e6-fa23-4fbe-bf57-6f609930c105-c000.csv
uploaded.part-00001-b538a6e6-fa23-4fbe-bf57-6f609930c105-c000.csv.crc
uploaded.part-00000-b538a6e6-fa23-4fbe-bf57-6f609930c105-c000.csv.crc
uploaded_SUCCESS
uploaded._SUCCESS.crc


In [ ]:
gold_path = "/content/drive/MyDrive/Colab Notebooks/log_analytics/gold"

for file in os.listdir(gold_path):
  local_file = os.path.join(
      gold_path,
      file
  )

  if os.path.isfile(local_file):
    s3.upload_file(
        local_file,
        bucket_name,
        f"gold/{file}"
    )
    print(f"uploaded{file}")

uploadedpart-00000-490630dd-842b-4787-bb41-bdc1249f8125-c000.csv
uploaded.part-00000-490630dd-842b-4787-bb41-bdc1249f8125-c000.csv.crc
uploaded_SUCCESS
uploaded._SUCCESS.crc


SyntaxError: invalid syntax (432901036.py, line 1)